# Full Stack on Google Colab (Ollama + FastAPI + Streamlit)

Runs the **entire** Application Intelligence Platform in a single Colab VM:
Ollama (text + vision models) and the FastAPI backend stay purely internal
(`localhost`), and only the Streamlit portal is exposed publicly via a
Cloudflare quick tunnel.

**How this differs from [colab_ollama_server.ipynb](./colab_ollama_server.ipynb):**
that notebook only hosts a model server for a *separately-run local* copy of
this app. This notebook runs the whole app (backend + UI included) on Colab
itself - useful if you don't want to run anything locally at all.

> ⚠️ **Security note:** the tunnel URL is public with no additional
> authentication beyond the app's own login. Only use this for short-lived
> demos with synthetic/non-sensitive data, never real government/citizen
> data (see the Data Sovereignty section in the main README). Colab sessions
> disconnect after inactivity or ~12 hours, so treat this as ephemeral.

In [ ]:
REPO_URL = "https://github.com/Govindkm/tcs-ai-club-hackathon-prompt-pioneers.git"
BRANCH = "develop"
PROJECT_DIR = "/content/tcs-ai-club-hackathon-prompt-pioneers"

import os

if not os.path.isdir(PROJECT_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {PROJECT_DIR}
else:
    print("Repository already cloned - pulling latest changes.")
    !git -C {PROJECT_DIR} pull

%cd {PROJECT_DIR}

## 1. Install Python dependencies

In [ ]:
!pip install -q -r requirements.txt

## 2. Install and start Ollama (text + vision models)

`zstd` is required by the Ollama installer on Colab's base image.

In [ ]:
!sudo apt-get update -qq && sudo apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import os
import subprocess
import time

import requests

os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
ollama_process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):
    try:
        requests.get("http://127.0.0.1:11434", timeout=2)
        print("Ollama server is up.")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    raise RuntimeError("Ollama server did not start in time - check the logs and retry.")

MODEL_NAME = "llama3.2:3b"       # keep in sync with OLLAMA_MODEL_ID in the .env cell below
VISION_MODEL_NAME = "minicpm-v"  # keep in sync with OLLAMA_VISION_MODEL in the .env cell below
!ollama pull {MODEL_NAME}
!ollama pull {VISION_MODEL_NAME}

## 3. Configure the app and seed the database

Since Ollama, the backend, and Streamlit all run on this same VM, `OLLAMA_HOST`
and `API_BASE_URL` both point at `localhost` - no tunnels needed between them.

In [ ]:
import secrets
from pathlib import Path

Path(".env").write_text(f"""STRANDS_MODEL_PROVIDER=ollama
API_BASE_URL=http://localhost:8000
JWT_SECRET_KEY={secrets.token_hex(32)}
OLLAMA_HOST=http://localhost:11434
OLLAMA_MODEL_ID={MODEL_NAME}
OLLAMA_VISION_MODEL={VISION_MODEL_NAME}
STRANDS_TRACE_CONSOLE=true
STRANDS_TRACE_OTLP=false
OTEL_SERVICE_NAME=prompt-pioneers-poc
LOG_LEVEL=INFO
""")

!python scripts/seed_db.py

## 4. Start the FastAPI backend

In [ ]:
backend_process = subprocess.Popen(
    ["python", "-m", "uvicorn", "backend.app.main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

for _ in range(30):
    try:
        requests.get("http://127.0.0.1:8000/api/v1/health", timeout=2)
        print("Backend is up.")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    raise RuntimeError("Backend did not start in time - check for missing dependencies and retry.")

## 5. Start the Streamlit portal

In [ ]:
streamlit_process = subprocess.Popen(
    ["streamlit", "run", "streamlit_app.py", "--server.headless", "true", "--server.port", "8501"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

for _ in range(30):
    try:
        requests.get("http://127.0.0.1:8501", timeout=2)
        print("Streamlit is up.")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    raise RuntimeError("Streamlit did not start in time - check for missing dependencies and retry.")

## 6. Expose the Streamlit portal publicly

Only Streamlit (port 8501) needs a public tunnel - the backend and Ollama are
called locally from within this same VM.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

import re

tunnel_process = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
deadline = time.time() + 30
while time.time() < deadline:
    line = tunnel_process.stdout.readline()
    if not line:
        continue
    match = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if not public_url:
    raise RuntimeError("Could not find the tunnel URL yet - re-run this cell or check the logs.")

print(f"Public portal URL: {public_url}")

## 7. Log in

Open the public URL printed above and log in with the default admin seeded
in step 3:

- Username: `admin`
- Password: `ChangeMe123!`

Or register a new applicant account from the portal's Register tab. Keep this
notebook's runtime running for the duration of your demo - stopping it shuts
down Ollama, the backend, Streamlit, and the tunnel all at once.

## 8. Shutdown (run when you're done)

In [ ]:
for proc in [tunnel_process, streamlit_process, backend_process, ollama_process]:
    proc.terminate()
print("Tunnel, Streamlit, backend, and Ollama all stopped.")